In [24]:
import polars as pl
import math
from pathlib import Path

pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_width_chars(250)

polars.config.Config

In [25]:
import polars as pl
from pathlib import Path
from datetime import timedelta

In [26]:
GTFS = Path("raw/processed_gtfs")
RT = Path("raw")
 
# ---------------------------------------------------
# Read files
# ---------------------------------------------------
 
routes = pl.read_parquet(GTFS / "routes.parquet")
trips = pl.read_parquet(GTFS / "trips.parquet")
stops = pl.read_parquet(GTFS / "stops.parquet")
stop_times = pl.read_parquet(GTFS / "stop_times.parquet")
 
trip_updates = pl.read_parquet(RT / "trip_updates/2026-07-01.parquet")
vehicle_positions = pl.read_parquet(RT / "vehicle_positions/2026-07-01.parquet")
 
print("Loaded")

Loaded


In [27]:
def gtfs_to_seconds(col):
    p = pl.col(col).str.split(":")
    return (
        p.list.get(0).cast(pl.Int32) * 3600
        + p.list.get(1).cast(pl.Int32) * 60
        + p.list.get(2).cast(pl.Int32)
    )
 
stop_times = stop_times.with_columns(
    pl.col("stop_sequence").cast(pl.UInt32),
    pl.col("stop_id").cast(pl.Utf8),
    gtfs_to_seconds("arrival_time").alias("scheduled_arrival"),
    gtfs_to_seconds("departure_time").alias("scheduled_departure"),
)
 
stops = stops.with_columns([
    pl.col("stop_id").cast(pl.Utf8),

    # pl.col("stop_lat")
    #   .str.strip_chars()
    #   .cast(pl.Float64),

    # pl.col("stop_lon")
    #   .str.strip_chars()
    #   .cast(pl.Float64),
])

In [28]:

trip_updates = (
    trip_updates
    .sort("feed_timestamp")
    .group_by(["trip_id", "start_date", "stop_sequence"])
    .last()
)

In [29]:
if "schedule_relationship" in trip_updates.columns:
    trip_updates = trip_updates.filter(pl.col("schedule_relationship") == 0)
 

In [30]:
trip_updates = trip_updates.with_columns(
    pl.from_epoch("arrival_time", time_unit="s").alias("event_time")
)
 
vehicle_positions = vehicle_positions.with_columns(
    pl.from_epoch("timestamp", time_unit="s").alias("vehicle_time")
)

In [31]:
TRIP_ID_PATTERN = r'^[A-Z]{2}_([A-Z0-9]+)-(\w+?)-(\d+)_([A-Z0-9+]+)_(\d+)$'
 
def parse_trip_id(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns([
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 2).alias("_service_day"),
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 3).alias("_origin_secs"),
        pl.col("trip_id").str.extract(TRIP_ID_PATTERN, 5).alias("_trip_num"),
    ])
 
trips_parsed = parse_trip_id(trips)
tu_parsed = parse_trip_id(trip_updates)
 
# 1) direct trip_id match
direct = tu_parsed.join(
    trips_parsed.select(["trip_id", "route_id", "direction_id", "shape_id", "service_id"]),
    on="trip_id", how="inner", suffix="_static",
)

In [32]:
unmatched = tu_parsed.join(direct.select("trip_id").unique(), on="trip_id", how="anti")
fallback = unmatched.join(
    trips_parsed.select([
        "trip_id", "route_id", "direction_id", "shape_id", "service_id",
        "_service_day", "_origin_secs", "_trip_num",
    ]).rename({"trip_id": "_static_trip_id"}),
    on=["route_id", "_service_day", "_origin_secs", "_trip_num"],
    how="inner",
    suffix="_static",
).with_columns(
    pl.col("_static_trip_id").alias("trip_id")   # use the STATIC trip_id downstream
).drop("_static_trip_id")
 
n_direct, n_fallback, n_total = direct.height, fallback.height, tu_parsed.height
print(f"[join] direct match: {n_direct:,} | fallback match: {n_fallback:,} | "
      f"total: {(n_direct + n_fallback) / n_total:.1%} of {n_total:,} rows")
 
direct = direct.drop(["_service_day", "_origin_secs", "_trip_num"])
fallback = fallback.drop(["_service_day", "_origin_secs", "_trip_num"])

[join] direct match: 70,053 | fallback match: 0 | total: 100.0% of 70,053 rows


In [33]:
direct = (
    direct.drop(["route_id", "direction_id"])
          .rename({"route_id_static": "route_id", "direction_id_static": "direction_id"})
)
fallback = fallback.drop("direction_id").rename({"direction_id_static": "direction_id"})
 
matched = pl.concat([direct, fallback.select(direct.columns)])

In [34]:
data = (
    matched
    .join(
        stop_times.select([
            "trip_id", "stop_sequence", "stop_id",
            "scheduled_arrival", "scheduled_departure",
        ]),
        on=["trip_id", "stop_sequence"],
        how="inner",   # inner now: every row here already has a resolved static trip_id,
                        # so a missing stop_times row means bad data, not an expected gap
    )
    .join(
        stops.select(["stop_id", "stop_lat", "stop_lon"]),
        on="stop_id",
        how="left",
    )
)
 
print(data.shape)

(70053, 22)


In [35]:
vp = vehicle_positions.select([
    "vehicle_id", "vehicle_time", "latitude", "longitude", "bearing",
])
 
data = data.sort(["vehicle_id", "event_time"])
vp = vp.sort(["vehicle_id", "vehicle_time"])
 
data = data.join_asof(
    vp,
    left_on="event_time",
    right_on="vehicle_time",
    by="vehicle_id",
    strategy="backward",
    tolerance=timedelta(minutes=2),
)

C:\Users\ishan\AppData\Local\Temp\ipykernel_8604\1183410916.py:8: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  data = data.join_asof(


In [36]:
data = (
    data
    .sort(["trip_id", "stop_sequence"])
    .with_columns(
        pl.col("arrival_time").shift(-1).over("trip_id").alias("next_arrival_time")
    )
    .with_columns(
        (pl.col("next_arrival_time") - pl.col("arrival_time")).alias("travel_time")
    )
)
 
data = data.filter(
    (pl.col("travel_time") > 0) & (pl.col("travel_time") < 1800)   # 30 min cap
)

In [37]:
# data = data.with_columns([
#     pl.col("event_time").dt.hour().alias("hour"),
#     pl.col("event_time").dt.weekday().alias("weekday"),
#     pl.col("event_time").dt.month().alias("month"),
# ])
 
# data = data.with_columns([
#     (
#         (pl.col("hour").is_between(7, 9)) |
#         (pl.col("hour").is_between(16, 18))
#     ).cast(pl.Int8).alias("is_peak")
# ])

data = data.with_columns(
    pl.col("event_time")
      .dt.replace_time_zone("UTC")
      .dt.convert_time_zone("America/New_York")
      .alias("event_time_local")
)
 
data = data.with_columns([
    pl.col("event_time_local").dt.hour().alias("hour"),
    pl.col("event_time_local").dt.weekday().alias("weekday"),
    pl.col("event_time_local").dt.month().alias("month"),
])
 
data = data.with_columns([
    (
        (pl.col("hour").is_between(7, 9)) |
        (pl.col("hour").is_between(16, 18))
    ).cast(pl.Int8).alias("is_peak")
])

In [38]:
data = data.with_columns([
    (
        pl.col("scheduled_arrival")
        - pl.col("scheduled_departure").shift(1).over("trip_id")
    ).alias("scheduled_segment_time"),
 
    (
        pl.col("stop_sequence") / pl.col("stop_sequence").max().over("trip_id")
    ).alias("trip_progress"),
])

In [39]:
segment_network = pl.read_parquet("processed/segment_network.parquet")
 
# dtype fix: segment_network's stop_id/next_stop_id are Int64 (straight
# from stop_times.txt), but `data`'s stop_id was cast to Utf8 earlier for
# the RT join - cast both sides to match before joining.
segment_network = segment_network.with_columns([
    pl.col("stop_id").cast(pl.Utf8),
    pl.col("next_stop_id").cast(pl.Utf8),
])

In [40]:
data = (
    data
    .sort(["trip_id", "stop_sequence"])
    .with_columns(
        pl.col("stop_id").shift(-1).over("trip_id").alias("next_stop_id")
    )
)
 
# %%
n_before = data.height
 
data = data.join(
    segment_network.select([
        "shape_id", "stop_id", "next_stop_id",
        "segment_length", "scheduled_travel_time",
    ]).rename({"scheduled_travel_time": "segment_scheduled_travel_time"}),
    on=["shape_id", "stop_id", "next_stop_id"],
    how="left",
)
 
print(f"rows before: {n_before:,} | after segment join: {data.height:,}")
print("null segment_length rows:", data.filter(pl.col("segment_length").is_null()).height)
 

rows before: 67,196 | after segment join: 67,196
null segment_length rows: 2949


In [41]:
data = data.with_columns(
    (pl.col("segment_length") / pl.col("segment_scheduled_travel_time").clip(lower_bound=1))
    .alias("scheduled_segment_speed_mps")
)

In [42]:
data = data.sort(["trip_id", "stop_sequence"])
check = data.select([
    "trip_id", "stop_sequence", "scheduled_segment_time", "segment_scheduled_travel_time",
]).with_columns(
    # bring segment_scheduled_travel_time from row k up to align with
    # scheduled_segment_time at row k+1 (both now describe segment k->k+1)
    pl.col("segment_scheduled_travel_time").shift(1).over("trip_id").alias("segment_scheduled_travel_time_aligned")
).drop_nulls(subset=["scheduled_segment_time", "segment_scheduled_travel_time_aligned"])
 
diff = (check["scheduled_segment_time"] - check["segment_scheduled_travel_time_aligned"]).abs()
print("median abs diff (aligned):", diff.median())
print("rows with diff > 30s (aligned):", (diff > 30).sum(), "/", check.height)
 
# expected: last stop of each trip has null segment_length/next_stop_id -
# should be roughly one per trip, not a bug
n_null_segment = data.filter(pl.col("segment_length").is_null()).height
n_trips = data["trip_id"].n_unique()
print(f"null segment_length rows: {n_null_segment:,} vs trip count: {n_trips:,} (should be close)")
 

median abs diff (aligned): 20.0
rows with diff > 30s (aligned): 21580 / 64247
null segment_length rows: 2,949 vs trip count: 1,314 (should be close)


In [43]:

import requests
from datetime import datetime
 
STATION_LAT, STATION_LON = 40.7829, -73.9654  # Central Park
 
dates_needed = data["start_date"].unique().sort().to_list()
start_date = str(dates_needed[0])
end_date = str(dates_needed[-1])
start_fmt = f"{start_date[:4]}-{start_date[4:6]}-{start_date[6:8]}"
end_fmt = f"{end_date[:4]}-{end_date[4:6]}-{end_date[6:8]}"
 
print(f"Fetching weather for {start_fmt} to {end_fmt}")
 
resp = requests.get(
    "https://archive-api.open-meteo.com/v1/archive",
    params={
        "latitude": STATION_LAT,
        "longitude": STATION_LON,
        "start_date": start_fmt,
        "end_date": end_fmt,
        "hourly": "temperature_2m,precipitation,rain,snowfall,"
                  "windspeed_10m,weathercode",
        "timezone": "America/New_York",
    },
    timeout=30,
)
resp.raise_for_status()
weather_json = resp.json()["hourly"]
 
weather = pl.DataFrame({
    "weather_time_str": weather_json["time"],
    "temperature_c": weather_json["temperature_2m"],
    "precipitation_mm": weather_json["precipitation"],
    "rain_mm": weather_json["rain"],
    "snowfall_cm": weather_json["snowfall"],
    "windspeed_kmh": weather_json["windspeed_10m"],
    "weathercode": weather_json["weathercode"],
})
 
weather = weather.with_columns(
    pl.col("weather_time_str").str.strptime(pl.Datetime, "%Y-%m-%dT%H:%M").alias("weather_time")
).with_columns([
    pl.col("weather_time").dt.strftime("%Y%m%d").alias("start_date"),  # keep as str - matches data's dtype
    pl.col("weather_time").dt.hour().alias("hour"),
])
 
print(weather.shape)
print(weather.head())

Fetching weather for 2026-06-30 to 2026-07-01
(48, 10)
shape: (5, 10)
┌──────────────────┬───────────────┬──────────────────┬─────────┬─────────────┬───────────────┬─────────────┬─────────────────────┬────────────┬──────┐
│ weather_time_str ┆ temperature_c ┆ precipitation_mm ┆ rain_mm ┆ snowfall_cm ┆ windspeed_kmh ┆ weathercode ┆ weather_time        ┆ start_date ┆ hour │
│ ---              ┆ ---           ┆ ---              ┆ ---     ┆ ---         ┆ ---           ┆ ---         ┆ ---                 ┆ ---        ┆ ---  │
│ str              ┆ f64           ┆ f64              ┆ f64     ┆ f64         ┆ f64           ┆ i64         ┆ datetime[μs]        ┆ str        ┆ i8   │
╞══════════════════╪═══════════════╪══════════════════╪═════════╪═════════════╪═══════════════╪═════════════╪═════════════════════╪════════════╪══════╡
│ 2026-06-30T00:00 ┆ 24.6          ┆ 0.0              ┆ 0.0     ┆ 0.0         ┆ 5.7           ┆ 1           ┆ 2026-06-30 00:00:00 ┆ 20260630   ┆ 0    │
│ 2026-06-30T01:00

In [44]:
# %%
# ---------------------------------------------------------------------
# Join weather onto data by (start_date, hour) - both now NYC-local
# ---------------------------------------------------------------------
 
n_before = data.height
 
data = data.join(
    weather.select([
        "start_date", "hour", "temperature_c", "precipitation_mm",
        "rain_mm", "snowfall_cm", "windspeed_kmh", "weathercode",
    ]),
    on=["start_date", "hour"],
    how="left",
)
 
print(f"rows before: {n_before:,} | after weather join: {data.height:,}")
print("null weather rows:", data.filter(pl.col("temperature_c").is_null()).height)

rows before: 67,196 | after weather join: 67,196
null weather rows: 0


In [45]:
data = data.with_columns([
    (pl.col("precipitation_mm") > 0.1).cast(pl.Int8).alias("is_raining"),
    (pl.col("snowfall_cm") > 0.0).cast(pl.Int8).alias("is_snowing"),
    # WMO weather codes 45 (fog) / 48 (depositing rime fog) as a
    # low-visibility proxy - the archive API doesn't expose visibility
    # directly (ERA5 reanalysis has no such variable; that's a
    # forecast-API-only field), so this is the closest available signal.
    pl.col("weathercode").is_in([45, 48]).cast(pl.Int8).alias("is_fog"),
])

In [46]:
data

trip_id,start_date,stop_sequence,feed_timestamp,fetch_timestamp,start_time,schedule_relationship,vehicle_id,trip_timestamp,stop_id,arrival_time,departure_time,event_time,route_id,direction_id,shape_id,service_id,stop_id_right,scheduled_arrival,scheduled_departure,stop_lat,stop_lon,vehicle_time,latitude,longitude,bearing,next_arrival_time,travel_time,event_time_local,hour,weekday,month,is_peak,scheduled_segment_time,trip_progress,next_stop_id,segment_length,segment_scheduled_travel_time,scheduled_segment_speed_mps,temperature_c,precipitation_mm,rain_mm,snowfall_cm,windspeed_kmh,weathercode,is_raining,is_snowing,is_fog
str,str,u32,u64,"datetime[μs, UTC]",str,i32,str,u64,str,i64,i64,datetime[μs],str,i64,str,str,str,i32,i32,f64,f64,datetime[μs],f32,f32,f32,i64,i64,"datetime[μs, America/New_York]",i8,i8,i8,i8,i32,f64,str,f64,i64,f64,f64,f64,f64,f64,f64,i64,i8,i8,i8
"""MV_C6-Weekday-110900_M4_447""","""20260630""",70,1782864003,2026-07-01 00:00:28.366841 UTC,"""""",0,"""MTA NYCT_9760""",1782863994,"""400629""",1782864018,1782864018,2026-07-01 00:00:18,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400629""",71825,71825,40.847906,-73.939407,2026-06-30 23:59:54,40.847366,-73.939713,67.574135,1782864104,86,2026-06-30 20:00:18 EDT,20,2,6,0,null,0.921053,"""400630""",153.41805,35,4.383373,31.1,0.0,0.0,0.0,9.5,2,0,0,0
"""MV_C6-Weekday-110900_M4_447""","""20260630""",71,1782864107,2026-07-01 00:01:58.841053 UTC,"""""",0,"""MTA NYCT_9760""",1782864084,"""400630""",1782864104,1782864104,2026-07-01 00:01:44,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400630""",71869,71869,40.849229,-73.938932,2026-07-01 00:01:24,40.849068,-73.939072,61.927513,1782864181,77,2026-06-30 20:01:44 EDT,20,2,6,0,44,0.934211,"""400631""",230.315978,52,4.429153,31.1,0.0,0.0,0.0,9.5,2,0,0,0
"""MV_C6-Weekday-110900_M4_447""","""20260630""",72,1782864180,2026-07-01 00:03:28.367952 UTC,"""""",0,"""MTA NYCT_9760""",1782864174,"""400631""",1782864181,1782864181,2026-07-01 00:03:01,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400631""",71934,71934,40.851202,-73.938078,2026-07-01 00:02:54,40.851707,-73.937981,69.819817,1782864238,57,2026-06-30 20:03:01 EDT,20,2,6,0,65,0.947368,"""400632""",137.22478,31,4.426606,31.1,0.0,0.0,0.0,9.5,2,0,0,0
"""MV_C6-Weekday-110900_M4_447""","""20260630""",73,1782864253,2026-07-01 00:04:28.367085 UTC,"""""",0,"""MTA NYCT_9760""",1782864234,"""400632""",1782864238,1782864238,2026-07-01 00:03:58,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400632""",71973,71973,40.852396,-73.937654,2026-07-01 00:03:54,40.852718,-73.93763,72.790443,1782864275,37,2026-06-30 20:03:58 EDT,20,2,6,0,39,0.960526,"""400633""",141.957749,32,4.43618,31.1,0.0,0.0,0.0,9.5,2,0,0,0
"""MV_C6-Weekday-110900_M4_447""","""20260630""",74,1782864285,2026-07-01 00:04:58.686397 UTC,"""""",0,"""MTA NYCT_9760""",1782864264,"""400633""",1782864275,1782864275,2026-07-01 00:04:35,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400633""",72013,72013,40.853639,-73.937266,2026-07-01 00:04:24,40.85331,-73.937447,72.790443,1782864328,53,2026-06-30 20:04:35 EDT,20,2,6,0,40,0.973684,"""400634""",165.220801,38,4.347916,31.1,0.0,0.0,0.0,9.5,2,0,0,0
"""MV_C6-Weekday-110900_M4_447""","""20260630""",75,1782864337,2026-07-01 00:05:58.545280 UTC,"""""",0,"""MTA NYCT_9760""",1782864324,"""400634""",1782864328,1782864328,2026-07-01 00:05:28,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400634""",72060,72060,40.85507,-73.936723,2026-07-01 00:05:24,40.855469,-73.936768,61.821411,1782864426,98,2026-06-30 20:05:28 EDT,20,2,6,0,47,0.986842,"""400635""",282.623586,64,4.415994,31.1,0.0,0.0,0.0,9.5,2,0,0,0
"""MV_C6-Weekday-110900_M4_447""","""20260630""",76,1782864430,2026-07-01 00:07:28.367644 UTC,"""""",0,"""MTA NYCT_9760""",1782864414,"""400635""",1782864426,1782864426,2026-07-01 00:07:06,"""M4""",0,"""M040923""","""MV_C6-Weekday""","""400635""",72140,72140,40.857335,-73.935335,2026-07-01 00:06:54,40.857254,-73.935486,53.445103,1782864898,472,2026-06-30 20:07:06 EDT,20,2,6,0,80,1.0,null,null,null,

In [47]:
features = [
    "route_id",
    "direction_id",
    "shape_id",
    "service_id",
 
    "stop_sequence",
    "trip_progress",
 
    "hour",
    "weekday",
    "month",
    "is_peak",
 
    "scheduled_arrival",
    "scheduled_departure",
    "scheduled_segment_time",
 
    "stop_lat",
    "stop_lon",
 
    "latitude",
    "longitude",
    "bearing",
    "temperature_c",
    "precipitation_mm",
    "snowfall_cm",
    "windspeed_kmh",
    "is_raining",
    "is_snowing",
    "is_fog",
    "weathercode",
    "segment_length",
    "scheduled_segment_speed_mps",
]
target = "travel_time"
id_cols = ["trip_id", "start_date"]
 
print(data.columns)
 
# %%
model_ready = data.select(id_cols + features + [target]).drop_nulls(subset=features + [target])
 
print(model_ready.shape)
print(model_ready.null_count())
print(model_ready.head())

['trip_id', 'start_date', 'stop_sequence', 'feed_timestamp', 'fetch_timestamp', 'start_time', 'schedule_relationship', 'vehicle_id', 'trip_timestamp', 'stop_id', 'arrival_time', 'departure_time', 'event_time', 'route_id', 'direction_id', 'shape_id', 'service_id', 'stop_id_right', 'scheduled_arrival', 'scheduled_departure', 'stop_lat', 'stop_lon', 'vehicle_time', 'latitude', 'longitude', 'bearing', 'next_arrival_time', 'travel_time', 'event_time_local', 'hour', 'weekday', 'month', 'is_peak', 'scheduled_segment_time', 'trip_progress', 'next_stop_id', 'segment_length', 'segment_scheduled_travel_time', 'scheduled_segment_speed_mps', 'temperature_c', 'precipitation_mm', 'rain_mm', 'snowfall_cm', 'windspeed_kmh', 'weathercode', 'is_raining', 'is_snowing', 'is_fog']
(57341, 31)
shape: (1, 31)
┌────────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬──────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬─

In [48]:
model_ready.columns

['trip_id',
 'start_date',
 'route_id',
 'direction_id',
 'shape_id',
 'service_id',
 'stop_sequence',
 'trip_progress',
 'hour',
 'weekday',
 'month',
 'is_peak',
 'scheduled_arrival',
 'scheduled_departure',
 'scheduled_segment_time',
 'stop_lat',
 'stop_lon',
 'latitude',
 'longitude',
 'bearing',
 'temperature_c',
 'precipitation_mm',
 'snowfall_cm',
 'windspeed_kmh',
 'is_raining',
 'is_snowing',
 'is_fog',
 'weathercode',
 'segment_length',
 'scheduled_segment_speed_mps',
 'travel_time']

In [49]:
print(model_ready.describe())

shape: (9, 32)
┌─────┬─────┬─────┬─────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ sta ┆ tri ┆ sta ┆ rou ┆ direct ┆ shape ┆ servi ┆ stop_ ┆ trip_ ┆ hour  ┆ weekd ┆ month ┆ is_pe ┆ sched ┆ sched ┆ sched ┆ stop_ ┆ stop_ ┆ latit ┆ longi ┆ beari ┆ tempe ┆ preci ┆ snowf ┆ winds ┆ is_ra ┆ is_sn ┆ is_fo ┆ weath ┆ segme ┆ sched ┆ trave │
│ tis ┆ p_i ┆ rt_ ┆ te_ ┆ ion_id ┆ _id   ┆ ce_id ┆ seque ┆ progr ┆ ---   ┆ ay    ┆ ---   ┆ ak    ┆ uled_ ┆ uled_ ┆ uled_ ┆ lat   ┆ lon   ┆ ude   ┆ tude  ┆ ng    ┆ ratur ┆ pitat ┆ all_c ┆ peed_ ┆ ining ┆ owing ┆ g     ┆ ercod ┆ nt_le ┆ uled_ ┆ l_tim │
│ tic ┆ d   ┆ dat ┆ id  ┆ ---    ┆ ---   ┆ ---   ┆ nce   ┆ ess   ┆ f64   ┆ ---   ┆ f64   ┆ ---   ┆ arriv ┆ depar ┆ segme ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ e_c   ┆ ion_m ┆ m     ┆ kmh   ┆ ---   ┆ ---   ┆ ---   ┆ e     ┆ ngth 

In [50]:
model_ready.group_by("route_id").len().sort("len", descending=True)

route_id,len
str,u32
"""M4""",14142
"""M101""",12548
"""M15""",11161
"""M1""",10779
"""M2""",8711


In [51]:
model_ready.select([
    pl.col("scheduled_arrival").min().alias("min"),
    pl.col("scheduled_arrival").max().alias("max"),
    pl.col("scheduled_arrival").mean().alias("mean"),
])

min,max,mean
i32,i32,f64
1241,95134,51804.658098


In [52]:
print(
    trip_updates.group_by("route_id").len().sort("len", descending=True)
)

print(
    matched.group_by("route_id").len().sort("len", descending=True)
)

print(
    model_ready.group_by("route_id").len().sort("len", descending=True)
)

shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 16698 │
│ M101     ┆ 15665 │
│ M15      ┆ 14044 │
│ M1       ┆ 13106 │
│ M2       ┆ 10540 │
└──────────┴───────┘
shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 16698 │
│ M101     ┆ 15665 │
│ M15      ┆ 14044 │
│ M1       ┆ 13106 │
│ M2       ┆ 10540 │
└──────────┴───────┘
shape: (5, 2)
┌──────────┬───────┐
│ route_id ┆ len   │
│ ---      ┆ ---   │
│ str      ┆ u32   │
╞══════════╪═══════╡
│ M4       ┆ 14142 │
│ M101     ┆ 12548 │
│ M15      ┆ 11161 │
│ M1       ┆ 10779 │
│ M2       ┆ 8711  │
└──────────┴───────┘


In [53]:
model_ready.write_parquet("raw/processed_gtfs/baseline_dataset.parquet")
print("written")

written
